# Step 2 - Does the training loop run?

Before we put the real reward in, we check the training machinery runs at all.
This trains for a few steps with a FAKE reward (a random number). We are not
looking for the model to get better, only for it to run start to finish without
crashing.

We use plain TRL, not Unsloth. Unsloth's current version has a bug in its GRPO
trainer. TRL is the standard library underneath and it works. See
docs/deviation_log.md D-007.

IMPORTANT: do not import unsloth anywhere in this notebook. Importing it
silently swaps TRL's trainer for Unsloth's broken one. If you imported unsloth
earlier in this session, use Runtime > Restart session, then run this notebook
from the top.

## Setup: get the repo

In [17]:
import os, sys, subprocess

REPO = "https://github.com/ookino/rlvr-argument-mining.git"
NAME = "rlvr-argument-mining"

if os.path.basename(os.getcwd()) != NAME:
    if not os.path.isdir(NAME):
        subprocess.run(["git", "clone", REPO], check=True)
    os.chdir(NAME)

# Force the code to match the remote. Running a notebook writes outputs back
# into the .ipynb, which counts as a local change and blocks a normal pull.
# reset --hard throws those away so the runtime always has the latest code.
subprocess.run(["git", "fetch", "--quiet"], check=False)
subprocess.run(["git", "reset", "--hard", "origin/main"], check=False)

sys.path.insert(0, os.getcwd())
print("repo ready, at", os.getcwd())

repo ready, at /content/rlvr-argument-mining


## Make sure the training libraries are present

These usually come pre-installed on Colab. This just checks they are there. It does not touch unsloth.

In [14]:
# Make sure the core libraries are present. Usually already on Colab.
!pip install -q trl peft bitsandbytes accelerate datasets

## Print the library versions

If the training cell fails later, these are the first thing to check. Note we do
NOT import unsloth here, on purpose.

In [15]:
import torch, trl, transformers, peft, bitsandbytes
print("trl         ", trl.__version__)
print("transformers", transformers.__version__)
print("peft        ", peft.__version__)
print("bitsandbytes", bitsandbytes.__version__)
print("torch       ", torch.__version__)
print("gpu         ", torch.cuda.is_available())

trl          0.24.0
transformers 5.5.0
peft         0.19.1
bitsandbytes 0.50.0
torch        2.11.0+cu128
gpu          True


## Run the smoke test

This loads Qwen 2.5 3B in 4-bit, adds the small trainable adapters, and runs 5
GRPO steps on a handful of toy questions with a random reward. Loading the model
takes a minute. The steps are slow because the model writes several answers per
question, and without a generation speed-up this is the plain path.

Success looks like a short training table and the line:

    training loop finished without crashing

In [16]:
from train.grpo_train import run

trainer = run("configs/baseline.yaml", max_steps=5)

RuntimeError: Failed to import trl.trainer.grpo_trainer because of the following error (look up to see its traceback):
No module named 'mergekit'

## What to do next

- If it printed **training loop finished without crashing**: step 2 is done.
  Tell Claude and we wire in the real reward.
- If it **errored**: copy the whole error and paste it back.

In [10]:
!git pull


Already up to date.
